# STEM Audio Workbench — Free On-Demand Runner

Run this notebook in **Google Colab**. It installs the open-source Demucs separation engine, lets you upload a song, previews the six estimated stems, and creates a ZIP containing WAV + MP3 versions aligned to the source duration.

**Free-tier note:** processing runs only while your Colab session is active. Free GPU availability and session limits are controlled by Google; CPU mode may be slow. Do not close the session while processing. The temporary share link is not a permanent hosted service.

Only upload audio you own or are authorised to process.

In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip -q install demucs==4.0.1 gradio==5.50.0


In [ ]:
import os, sys, shutil, subprocess, tempfile, json
from pathlib import Path
import gradio as gr

REPO = Path('/content/STEM')
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','https://github.com/demtaiwo1-eng/Stem.git',str(REPO)], check=True)
WORK = Path('/content/stem-work')
WORK.mkdir(exist_ok=True)
DEVICE = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
print('STEM ready. Compute device:', DEVICE)


In [ ]:
def separate_audio(audio_path, progress=gr.Progress()):
    if not audio_path:
        raise gr.Error('Upload an audio file first.')
    src = Path(audio_path)
    job_id = 'colab-' + next(tempfile._get_candidate_names())
    job_dir = WORK / job_id
    job_dir.mkdir(parents=True, exist_ok=True)
    safe_input = job_dir / ('source' + (src.suffix.lower() or '.audio'))
    shutil.copy2(src, safe_input)
    progress(0.05, desc='Preparing audio')
    env = os.environ.copy()
    env['STEM_DEVICE'] = DEVICE
    env['STEM_MODEL'] = 'htdemucs_6s'
    proc = subprocess.Popen([sys.executable, str(REPO / 'worker.py'), str(safe_input), str(job_dir), job_id], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, bufsize=1)
    lines = []
    for line in proc.stdout:
        line = line.strip()
        if line:
            lines.append(line)
            print(line)
            progress(min(0.95, 0.12 + 0.02 * len(lines)), desc=line[:90])
    code = proc.wait()
    if code != 0:
        raise gr.Error('Separation failed. Review the notebook output above for the Demucs error.')
    out = job_dir / 'deliverables'
    manifest = json.loads((out / 'manifest.json').read_text())
    by_name = {item['name']: str(out / item['mp3']) for item in manifest['stems']}
    def get(name): return by_name.get(name)
    zip_path = str(job_dir / 'stem-results.zip')
    progress(1, desc='Your stem pack is ready')
    return get('vocals'), get('drums'), get('bass'), get('guitar'), get('piano'), get('other'), zip_path, json.dumps(manifest, indent=2)


In [ ]:
with gr.Blocks(title='STEM Audio Workbench') as demo:
    gr.Markdown('# STEM Audio Workbench\nUpload a song and separate it into six aligned audio layers. The first run downloads the AI model and can take several minutes.')
    audio = gr.File(label='Choose an audio file', file_types=['audio'], type='filepath')
    run = gr.Button('Separate my audio', variant='primary')
    with gr.Accordion('Listen to individual stems', open=True):
        vocals = gr.Audio(label='Vocals', type='filepath')
        drums = gr.Audio(label='Drums', type='filepath')
        bass = gr.Audio(label='Bass', type='filepath')
        guitar = gr.Audio(label='Guitar', type='filepath')
        piano = gr.Audio(label='Piano', type='filepath')
        other = gr.Audio(label='Other accompaniment', type='filepath')
    result_zip = gr.File(label='Download all stems (ZIP)')
    details = gr.Code(label='Separation manifest', language='json')
    run.click(separate_audio, inputs=audio, outputs=[vocals, drums, bass, guitar, piano, other, result_zip, details])
    gr.Markdown('**Export:** stereo 44.1 kHz WAV and 320 kbps MP3, with each stem trimmed/padded to the source duration. AI source separation is approximate; some bleed and artifacts are possible. The `other` stem contains remaining accompaniment.')

demo.queue(max_size=2).launch(share=True, debug=True)
